# **Projet Steam: bloc2 Big Data**

## **PART1: LOADING AND CLEANING**

In [0]:
from pyspark.sql import functions as F

path = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
df = spark.read.json(path)

df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:
df.count()
# il ya 55 691 lignes

55691

In [0]:
df.show(5)
# tout est dans la colonne data

+--------------------+-------+
|                data|     id|
+--------------------+-------+
|{10, [Multi-playe...|     10|
|{1000000, [Single...|1000000|
|{1000010, [Single...|1000010|
|{1000030, [Multi-...|1000030|
|{1000040, [Single...|1000040|
+--------------------+-------+
only showing top 5 rows


## **printSchema**

In [0]:
df.schema.jsonValue()

{'type': 'struct',
 'fields': [{'name': 'data',
   'type': {'type': 'struct',
    'fields': [{'name': 'appid',
      'type': 'long',
      'nullable': True,
      'metadata': {}},
     {'name': 'categories',
      'type': {'type': 'array', 'elementType': 'string', 'containsNull': True},
      'nullable': True,
      'metadata': {}},
     {'name': 'ccu', 'type': 'long', 'nullable': True, 'metadata': {}},
     {'name': 'developer', 'type': 'string', 'nullable': True, 'metadata': {}},
     {'name': 'discount', 'type': 'string', 'nullable': True, 'metadata': {}},
     {'name': 'genre', 'type': 'string', 'nullable': True, 'metadata': {}},
     {'name': 'header_image',
      'type': 'string',
      'nullable': True,
      'metadata': {}},
     {'name': 'initialprice',
      'type': 'string',
      'nullable': True,
      'metadata': {}},
     {'name': 'languages', 'type': 'string', 'nullable': True, 'metadata': {}},
     {'name': 'name', 'type': 'string', 'nullable': True, 'metadata': {}},
 

In [0]:
df.schema.jsonValue().keys()
# Only two keys at this stage, type and fields, let's explore the type key

dict_keys(['type', 'fields'])

In [0]:
# Liste des 22 colonnes
# appid: long (nullable = true) 
# categories: array (nullable = true)     
# element: string (containsNull = true) 
# ccu: long (nullable = true) 
# developer: string (nullable = true) 
# discount: string (nullable = true) 
# genre: string (nullable = true) 
# header_image: string (nullable = true) 
# initialprice: string (nullable = true) 
# languages: string (nullable = true) 
# name: string (nullable = true) 
# negative: long (nullable = true) 
# owners: string (nullable = true) 
# platforms: struct (nullable = true) 
# positive: long (nullable = true) 
# price: string (nullable = true) 
# publisher: string (nullable = true) 
# release_date: string (nullable = true) 
# required_age: string (nullable = true) 
# short_description: string (nullable = true) 
# tags: struct (nullable = true) 
# type: string (nullable = true) 
# website: string (nullable = true)


In [0]:
df.schema.jsonValue()["type"]
# The value associated is struct

'struct'

In [0]:
# let's explore the content of the other key
df.schema.jsonValue()["fields"]

[{'name': 'data',
  'type': {'type': 'struct',
   'fields': [{'name': 'appid',
     'type': 'long',
     'nullable': True,
     'metadata': {}},
    {'name': 'categories',
     'type': {'type': 'array', 'elementType': 'string', 'containsNull': True},
     'nullable': True,
     'metadata': {}},
    {'name': 'ccu', 'type': 'long', 'nullable': True, 'metadata': {}},
    {'name': 'developer', 'type': 'string', 'nullable': True, 'metadata': {}},
    {'name': 'discount', 'type': 'string', 'nullable': True, 'metadata': {}},
    {'name': 'genre', 'type': 'string', 'nullable': True, 'metadata': {}},
    {'name': 'header_image',
     'type': 'string',
     'nullable': True,
     'metadata': {}},
    {'name': 'initialprice',
     'type': 'string',
     'nullable': True,
     'metadata': {}},
    {'name': 'languages', 'type': 'string', 'nullable': True, 'metadata': {}},
    {'name': 'name', 'type': 'string', 'nullable': True, 'metadata': {}},
    {'name': 'negative', 'type': 'long', 'nullable': T

In [0]:
# what keys does it have ?
df.schema.jsonValue()["fields"][0].keys()

dict_keys(['name', 'type', 'nullable', 'metadata'])

In [0]:
# the key name contains the name of the field
df.schema.jsonValue()["fields"][0]["name"]

'data'

In [0]:
# if we have the key type then we have subfields inside it
df.schema.jsonValue()["fields"][0]["type"]
# and we are back to the same structure we had at the beginning and we can start digging again
# that's the spirit of the function below

{'type': 'struct',
 'fields': [{'name': 'appid',
   'type': 'long',
   'nullable': True,
   'metadata': {}},
  {'name': 'categories',
   'type': {'type': 'array', 'elementType': 'string', 'containsNull': True},
   'nullable': True,
   'metadata': {}},
  {'name': 'ccu', 'type': 'long', 'nullable': True, 'metadata': {}},
  {'name': 'developer', 'type': 'string', 'nullable': True, 'metadata': {}},
  {'name': 'discount', 'type': 'string', 'nullable': True, 'metadata': {}},
  {'name': 'genre', 'type': 'string', 'nullable': True, 'metadata': {}},
  {'name': 'header_image', 'type': 'string', 'nullable': True, 'metadata': {}},
  {'name': 'initialprice', 'type': 'string', 'nullable': True, 'metadata': {}},
  {'name': 'languages', 'type': 'string', 'nullable': True, 'metadata': {}},
  {'name': 'name', 'type': 'string', 'nullable': True, 'metadata': {}},
  {'name': 'negative', 'type': 'long', 'nullable': True, 'metadata': {}},
  {'name': 'owners', 'type': 'string', 'nullable': True, 'metadata': {

## **Fonction walkSchema**

In [0]:
from pyspark.sql.types import StructType, StructField
from typing import List, Dict, Generator, Union, Callable

# This is actually written like a scala function, we'll walk you through it
def walkSchema(schema: Union[StructType, StructField]) -> Generator[str, None, None]:
    """Explores a PySpark schema:
    
    schema: StructType | StructField
    
    Yield
    -----
    A generator of strings, the name of each field in the schema
    """
    
    # we define a function _walk that produces a string generator from
    # a dictionnary "schema_dct", and a string "prefix"
    def _walk(schema_dct: Dict['str', Union['str', list, dict]],
              prefix: str = "") -> Generator[str, None, None]:
        assert isinstance(prefix, str), "prefix should be a string" # check if prefix is a string
        
        # this function returns "name" if there's no prefix and "prefix.name" if prefix exists
        fullName: Callable[str, str] = lambda name: ( 
            name if not prefix else f"{prefix}.{name}")
        
        # we get the next name one level lower from the dictionnary
        name = schema_dct.get('name', '')
        
        # if the type is struct then we search for the fields key
        # if fields is there we apply the function again and dig one level deeper in
        # the schema and set a prefix
        if schema_dct['type'] == 'struct':
            assert 'fields' in schema_dct, (
                "It's a StructType, we should have some fields")
            for field in schema_dct['fields']:
                yield from _walk(field, prefix=prefix)
        # if we have a dict type and we can't find fields then we
        # dig one level deeper and apply the _walk function again
        elif isinstance(schema_dct['type'], dict):
            assert 'fields' not in schema_dct, (
                "We're missing some keys here")
            yield from _walk(schema_dct['type'], prefix=fullName(name))
        # If we finally reached the end and found a name we yield the full name
        elif name:
            yield fullName(name)
    
    yield from _walk(schema.jsonValue())

# yield as opposed to return, returns a result but does not stop the function from running, it keeps
# running even after returning one result.

In [0]:
#Call walkSchema(...) on our dataframe schema: col_names then print it out to the screen
col_names = walkSchema(df.schema)
col_names

<generator object walkSchema at 0xfff1e823d440>

In [0]:
#You should see an output similar to <generator object walkSchema at 0x7f9eb0e390c0>.
#It's a Python's generator, you can read more about it here.

#For now, you just have to know, that just like a python's list, a generator is also iterable, which means we can iterate over it with a for loop.

#for e in my_generator:
    # You can access each element of the generator here
#We'll give it a try, by printing out the values of our col_names.

#Iterate over the walked schema
#NOTE: give the name col_name to the iterating variable

In [0]:
for col_name in walkSchema(df.schema):
  print(col_name)

data.appid
data.ccu
data.developer
data.discount
data.genre
data.header_image
data.initialprice
data.languages
data.name
data.negative
data.owners
data.platforms.linux
data.platforms.mac
data.platforms.windows
data.positive
data.price
data.publisher
data.release_date
data.required_age
data.short_description
data.tags.1980s
data.tags.1990's
data.tags.2.5D
data.tags.2D
data.tags.2D Fighter
data.tags.2D Platformer
data.tags.360 Video
data.tags.3D
data.tags.3D Fighter
data.tags.3D Platformer
data.tags.3D Vision
data.tags.4 Player Local
data.tags.4X
data.tags.6DOF
data.tags.8-bit Music
data.tags.ATV
data.tags.Abstract
data.tags.Action
data.tags.Action RPG
data.tags.Action RTS
data.tags.Action Roguelike
data.tags.Action-Adventure
data.tags.Addictive
data.tags.Adventure
data.tags.Agriculture
data.tags.Aliens
data.tags.Alternate History
data.tags.Ambient
data.tags.America
data.tags.Animation & Modeling
data.tags.Anime
data.tags.Arcade
data.tags.Archery
data.tags.Arena Shooter
data.tags.Artific

In [0]:
#Perfect, that's all the leafs of our schema.
#And we can just repeat the work we did with items.snippet.title for every column of this list.

#There are a couple ways to do this, you've got at least 2 options (using standard "non-functionnal" python):

#build a list comprehension (or unpack the generator) and pass it to a .select(...) statement
#iterate over the generator, and use .withColumn(...)
#But our favorite uses a functional approach. It particularly makes sense because Spark is based on Scala, a functionnal language.
#If you're interested in this approach, take a look at reduce from the functools package in Python.
#In this simple isolated case, it actually makes things look a bit harder than they should, but it would make it easier to neatly integrate this step in a global pipeline.
#Beware, if you're not familiar with functional programming that will probably feel non-trivial.

In [0]:
# Flatten du schéma
# 1. Récupérer dynamiquement toutes les colonnes tags.*
tag_columns = [f"data.tags.`{c}`" for c in df.select("data.tags.*").columns]

# 2. Flatten du schéma
df_flat = df.select(
    "data.appid",
    "data.name",
    "data.developer",
    "data.publisher",
    "data.price",
    "data.initialprice",
    "data.discount",
    "data.positive",
    "data.negative",
    "data.genre",
    "data.languages",
    "data.categories",
    "data.release_date",
    "data.required_age",
    "data.short_description",
    "data.owners",
    "data.ccu",
    "data.header_image",
    "data.type",
    "data.website",

    # plateformes
    "data.platforms.windows",
    "data.platforms.mac",
    "data.platforms.linux",

    # tags (toutes les colonnes tags.*)
    *tag_columns
)

display(df_flat.limit(5))

appid,name,developer,publisher,price,initialprice,discount,positive,negative,genre,languages,categories,release_date,required_age,short_description,owners,ccu,header_image,type,website,windows,mac,linux,1980s,1990's,2.5D,2D,2D Fighter,2D Platformer,360 Video,3D,3D Fighter,3D Platformer,3D Vision,4 Player Local,4X,6DOF,8-bit Music,ATV,Abstract,Action,Action RPG,Action RTS,Action Roguelike,Action-Adventure,Addictive,Adventure,Agriculture,Aliens,Alternate History,Ambient,America,Animation & Modeling,Anime,Arcade,Archery,Arena Shooter,Artificial Intelligence,Assassin,Asymmetric VR,Asynchronous Multiplayer,Atmospheric,Audio Production,Auto Battler,Automation,Automobile Sim,BMX,Base-Building,Baseball,Based On A Novel,Basketball,Battle Royale,Beat 'em up,Beautiful,Benchmark,Bikes,Blood,Board Game,Boss Rush,Bowling,Boxing,Building,Bullet Hell,Bullet Time,CRPG,Capitalism,Card Battler,Card Game,Cartoon,Cartoony,Casual,Cats,Character Action Game,Character Customization,Chess,Choices Matter,Choose Your Own Adventure,Cinematic,City Builder,Class-Based,Classic,Clicker,Co-op,Co-op Campaign,Coding,Cold War,Collectathon,Colony Sim,Colorful,Combat,Combat Racing,Comedy,Comic Book,Competitive,Conspiracy,Controller,Conversation,Cooking,Cozy,Crafting,Creature Collector,Cricket,Crime,Crowdfunded,Cult Classic,Cute,Cyberpunk,Cycling,Dark,Dark Comedy,Dark Fantasy,Dark Humor,Dating Sim,Deckbuilding,Demons,Design & Illustration,Destruction,Detective,Difficult,Dinosaurs,Diplomacy,Documentary,Dog,Dragons,Drama,Driving,Dungeon Crawler,Dungeons & Dragons,Dynamic Narration,Dystopian,Early Access,Economy,Education,Electronic,Electronic Music,Emotional,Epic,Episodic,Escape Room,Experience,Experimental,Exploration,FMV,FPS,Faith,Family Friendly,Fantasy,Farming,Farming Sim,Fast-Paced,Feature Film,Female Protagonist,Fighting,First-Person,Fishing,Flight,Football,Foreign,Free to Play,Funny,Futuristic,Gambling,Game Development,GameMaker,Games Workshop,Gaming,God Game,Golf,Gore,Gothic,Grand Strategy,Great Soundtrack,Grid-Based Movement,Gun Customization,Hack and Slash,Hacking,Hand-drawn,Hardware,Heist,Hentai,Hero Shooter,Hex Grid,Hidden Object,Historical,Hockey,Horror,Horses,Hunting,Idler,Illuminati,Immersive,Immersive Sim,Indie,Instrumental Music,Intentionally Awkward Controls,Interactive Fiction,Inventory Management,Investigation,Isometric,JRPG,Jet,Job Simulator,Jump Scare,Kickstarter,LEGO,LGBTQ+,Lemmings,Level Editor,Life Sim,Linear,Local Co-Op,Local Multiplayer,Logic,Loot,Looter Shooter,Lore-Rich,Lovecraftian,MMORPG,MOBA,Magic,Mahjong,Management,Mars,Martial Arts,Massively Multiplayer,Masterpiece,Match 3,Mature,Mechs,Medical Sim,Medieval,Memes,Metroidvania,Military,Mini Golf,Minigames,Minimalist,Mining,Mod,Moddable,Modern,Motocross,Motorbike,Mouse only,Movie,Multiplayer,Multiple Endings,Music,Music-Based Procedural Generation,Musou,Mystery,Mystery Dungeon,Mythology,NSFW,Narration,Narrative,Nature,Naval,Naval Combat,Ninja,Noir,Nonlinear,Nostalgia,Nudity,Offroad,Old School,On-Rails Shooter,Online Co-Op,Open World,Open World Survival Craft,Otome,Outbreak Sim,Parkour,Parody,Party,Party Game,Party-Based RPG,Perma Death,Philosophical,Photo Editing,Physics,Pinball,Pirates,Pixel Graphics,Platformer,Point & Click,Political,Political Sim,Politics,Pool,Post-apocalyptic,Precision Platformer,Procedural Generation,Programming,Psychedelic,Psychological,Psychological Horror,Puzzle,Puzzle-Platformer,PvE,PvP,Quick-Time Events,RPG,RPGMaker,RTS,Racing,Real Time Tactics,Real-Time,Real-Time with Pause,Realistic,Reboot,Relaxing,Remake,Replay Value,Resource Management,Retro,Rhythm,Robots,Rock Music,Rogue-like,Rogue-lite,Roguelike Deckbuilder,Roguevania,Romance,Rome,Rugby,Runner,Sailing,Sandbox,Satire,Sci-fi,Science,Score Attack,Sequel,Sexual Content,Shoot 'Em Up,Shooter,Shop Keeper,Short,Side Scroller,Silent Protagonist,Simulation,Singleplayer,Skateboarding,Skating,Skiing,Sniper,Snooker,Snow,Snowboarding,Soccer,Social Deduction,Software,Software Training,Sokoban,Solitaire,Souls-like,Sound

In [0]:
from functools import reduce
from pyspark.sql import functions as F

# 1. Exploration du schéma avec walkSchema
col_names = walkSchema(df_flat.schema)

print("Colonnes détectées dans le schéma :")
for col_name in col_names:
    print(col_name)

# 2. Nous n'utilisons PAS walkSchema pour reconstruire le DataFrame,
#    car certaines colonnes (ex: categories) sont des array<string>
#    et walkSchema essaie de créer des colonnes invalides (ex: '2D', '1980s').

df_clean_base = df_flat

display(df_clean_base.limit(5))


Colonnes détectées dans le schéma :
appid
name
developer
publisher
price
initialprice
discount
positive
negative
genre
languages
release_date
required_age
short_description
owners
ccu
header_image
type
website
windows
mac
linux
1980s
1990's
2.5D
2D
2D Fighter
2D Platformer
360 Video
3D
3D Fighter
3D Platformer
3D Vision
4 Player Local
4X
6DOF
8-bit Music
ATV
Abstract
Action
Action RPG
Action RTS
Action Roguelike
Action-Adventure
Addictive
Adventure
Agriculture
Aliens
Alternate History
Ambient
America
Animation & Modeling
Anime
Arcade
Archery
Arena Shooter
Artificial Intelligence
Assassin
Asymmetric VR
Asynchronous Multiplayer
Atmospheric
Audio Production
Auto Battler
Automation
Automobile Sim
BMX
Base-Building
Baseball
Based On A Novel
Basketball
Battle Royale
Beat 'em up
Beautiful
Benchmark
Bikes
Blood
Board Game
Boss Rush
Bowling
Boxing
Building
Bullet Hell
Bullet Time
CRPG
Capitalism
Card Battler
Card Game
Cartoon
Cartoony
Casual
Cats
Character Action Game
Character Customization
Ch

appid,name,developer,publisher,price,initialprice,discount,positive,negative,genre,languages,categories,release_date,required_age,short_description,owners,ccu,header_image,type,website,windows,mac,linux,1980s,1990's,2.5D,2D,2D Fighter,2D Platformer,360 Video,3D,3D Fighter,3D Platformer,3D Vision,4 Player Local,4X,6DOF,8-bit Music,ATV,Abstract,Action,Action RPG,Action RTS,Action Roguelike,Action-Adventure,Addictive,Adventure,Agriculture,Aliens,Alternate History,Ambient,America,Animation & Modeling,Anime,Arcade,Archery,Arena Shooter,Artificial Intelligence,Assassin,Asymmetric VR,Asynchronous Multiplayer,Atmospheric,Audio Production,Auto Battler,Automation,Automobile Sim,BMX,Base-Building,Baseball,Based On A Novel,Basketball,Battle Royale,Beat 'em up,Beautiful,Benchmark,Bikes,Blood,Board Game,Boss Rush,Bowling,Boxing,Building,Bullet Hell,Bullet Time,CRPG,Capitalism,Card Battler,Card Game,Cartoon,Cartoony,Casual,Cats,Character Action Game,Character Customization,Chess,Choices Matter,Choose Your Own Adventure,Cinematic,City Builder,Class-Based,Classic,Clicker,Co-op,Co-op Campaign,Coding,Cold War,Collectathon,Colony Sim,Colorful,Combat,Combat Racing,Comedy,Comic Book,Competitive,Conspiracy,Controller,Conversation,Cooking,Cozy,Crafting,Creature Collector,Cricket,Crime,Crowdfunded,Cult Classic,Cute,Cyberpunk,Cycling,Dark,Dark Comedy,Dark Fantasy,Dark Humor,Dating Sim,Deckbuilding,Demons,Design & Illustration,Destruction,Detective,Difficult,Dinosaurs,Diplomacy,Documentary,Dog,Dragons,Drama,Driving,Dungeon Crawler,Dungeons & Dragons,Dynamic Narration,Dystopian,Early Access,Economy,Education,Electronic,Electronic Music,Emotional,Epic,Episodic,Escape Room,Experience,Experimental,Exploration,FMV,FPS,Faith,Family Friendly,Fantasy,Farming,Farming Sim,Fast-Paced,Feature Film,Female Protagonist,Fighting,First-Person,Fishing,Flight,Football,Foreign,Free to Play,Funny,Futuristic,Gambling,Game Development,GameMaker,Games Workshop,Gaming,God Game,Golf,Gore,Gothic,Grand Strategy,Great Soundtrack,Grid-Based Movement,Gun Customization,Hack and Slash,Hacking,Hand-drawn,Hardware,Heist,Hentai,Hero Shooter,Hex Grid,Hidden Object,Historical,Hockey,Horror,Horses,Hunting,Idler,Illuminati,Immersive,Immersive Sim,Indie,Instrumental Music,Intentionally Awkward Controls,Interactive Fiction,Inventory Management,Investigation,Isometric,JRPG,Jet,Job Simulator,Jump Scare,Kickstarter,LEGO,LGBTQ+,Lemmings,Level Editor,Life Sim,Linear,Local Co-Op,Local Multiplayer,Logic,Loot,Looter Shooter,Lore-Rich,Lovecraftian,MMORPG,MOBA,Magic,Mahjong,Management,Mars,Martial Arts,Massively Multiplayer,Masterpiece,Match 3,Mature,Mechs,Medical Sim,Medieval,Memes,Metroidvania,Military,Mini Golf,Minigames,Minimalist,Mining,Mod,Moddable,Modern,Motocross,Motorbike,Mouse only,Movie,Multiplayer,Multiple Endings,Music,Music-Based Procedural Generation,Musou,Mystery,Mystery Dungeon,Mythology,NSFW,Narration,Narrative,Nature,Naval,Naval Combat,Ninja,Noir,Nonlinear,Nostalgia,Nudity,Offroad,Old School,On-Rails Shooter,Online Co-Op,Open World,Open World Survival Craft,Otome,Outbreak Sim,Parkour,Parody,Party,Party Game,Party-Based RPG,Perma Death,Philosophical,Photo Editing,Physics,Pinball,Pirates,Pixel Graphics,Platformer,Point & Click,Political,Political Sim,Politics,Pool,Post-apocalyptic,Precision Platformer,Procedural Generation,Programming,Psychedelic,Psychological,Psychological Horror,Puzzle,Puzzle-Platformer,PvE,PvP,Quick-Time Events,RPG,RPGMaker,RTS,Racing,Real Time Tactics,Real-Time,Real-Time with Pause,Realistic,Reboot,Relaxing,Remake,Replay Value,Resource Management,Retro,Rhythm,Robots,Rock Music,Rogue-like,Rogue-lite,Roguelike Deckbuilder,Roguevania,Romance,Rome,Rugby,Runner,Sailing,Sandbox,Satire,Sci-fi,Science,Score Attack,Sequel,Sexual Content,Shoot 'Em Up,Shooter,Shop Keeper,Short,Side Scroller,Silent Protagonist,Simulation,Singleplayer,Skateboarding,Skating,Skiing,Sniper,Snooker,Snow,Snowboarding,Soccer,Social Deduction,Software,Software Training,Sokoban,Solitaire,Souls-like,Sound